# HealthBrowse: evaluate your completed QLoRA adapter

**No retraining is needed.** Reuse the `llama3-general-research-002` run already saved in your Google Drive. This notebook compares the base Llama model and your adapter using HealthBrowse's browser/buddy prompts, then separately checks the original extraction test split.

The workload includes **12 browser/buddy questions, 2 reviewer controls and 25 extraction holdouts**, evaluated on both model routes (**78 generations**). Three additional interception cases are tested in the repository's CPU gateway tests and are never sent to the model. This is a development evaluation, not clinical certification or a public inference server.

1. Save a copy to your Drive. Choose **Runtime → Change runtime type → GPU**.
2. Keep your authorized Hugging Face read token in **Colab Secrets → HF_TOKEN**, with notebook access enabled. Never paste it into a cell.
3. Run the cells below in order. Keep the notebook open and monitor progress. GPU availability and completion are not guaranteed.
4. Your adapter, training data and training results are preserved. Evaluation output goes into a separate folder.

Do not run the previous training/comparison notebook in the same runtime. Start a fresh runtime for this evaluation. Only the included public/synthetic questions are used; do not add private patient information.

## 1. Install the pinned evaluation code

Setup uses a versioned source folder and removes incompatible optional vision/audio packages from this text-only Colab session. If it reports cached imports, select **Runtime → Restart session**, then run from the top. Do not delete your Drive run.

The repository must be accessible to your Git client. If it is private, use your institution's authorized access method; do not put GitHub tokens in notebook cells or clone URLs.

In [ ]:
from pathlib import Path
import subprocess, sys, json, os, tempfile, re
SOURCE_COMMIT = 'cc573995880e62755c7005f8a3a5c1d77179f77d'
SOURCE_URL = 'https://github.com/McChandra/medical_chatbot.git'
CACHE_ROOT = Path('/content')

def ensure_checkout(cache_root, commit, repository_url):
    if not re.fullmatch(r'[0-9a-f]{40}', commit):
        raise ValueError('Use a full source commit SHA.')
    cache_root = Path(cache_root)
    cache_root.mkdir(parents=True, exist_ok=True)
    destination = cache_root / ('healthbrowse-qlora-' + commit[:12])
    if not destination.exists():
        # Build in a new temporary directory; a failed fetch cannot leave a stale
        # final checkout. Never modify /content/medical_chatbot or saved Drive runs.
        with tempfile.TemporaryDirectory(prefix='hb-download-', dir=cache_root) as temporary:
            staging = Path(temporary) / 'repo'
            staging.mkdir()
            subprocess.run(['git','init','--quiet',str(staging)],check=True)
            subprocess.run(['git','-C',str(staging),'fetch','--quiet','--depth','1',repository_url,commit],check=True)
            subprocess.run(['git','-C',str(staging),'checkout','--quiet','--detach',commit],check=True)
            staging.rename(destination)
    if not (destination/'.git').is_dir():
        raise RuntimeError('The versioned path contains other files. Keep them and choose another CACHE_ROOT.')
    actual = subprocess.check_output(['git','-C',str(destination),'rev-parse','HEAD'],text=True).strip()
    if actual != commit:
        raise RuntimeError('The cached checkout has changed. Preserve it and choose another CACHE_ROOT.')
    if subprocess.check_output(['git','-C',str(destination),'status','--porcelain'],text=True).strip():
        raise RuntimeError('The cached checkout has local edits. Preserve them and choose another CACHE_ROOT.')
    return destination

REPO = ensure_checkout(CACHE_ROOT, SOURCE_COMMIT, SOURCE_URL)
INFERENCE = REPO / 'healthbrowse' / 'inference'
required = ['evaluate_adapter.py','evaluation_checks.py','adapter_artifact.py','requirements.txt']
missing = [name for name in required if not (INFERENCE/name).is_file()]
if missing:
    raise RuntimeError('Incomplete pinned source checkout: ' + ', '.join(missing))
SOURCE_VERSION = SOURCE_COMMIT
# A prior notebook attempt may have cached imports from the old checkout.
for name in ['evaluate_adapter','evaluation_checks','adapter_artifact','model_settings']:
    module = sys.modules.get(name)
    if module and Path(getattr(module,'__file__','')).parent != INFERENCE:
        del sys.modules[name]
print('Source version:', SOURCE_VERSION)
print('Source folder:', REPO)
print('Evaluation scripts: ready')
def prepare_text_environment(requirements):
    # This notebook evaluates text-only Llama. Colab's preinstalled image/audio
    # extensions may target a different torch ABI than the pinned torch 2.8.
    imported_before = [name for name in ['torch','transformers','torchvision','torchaudio','peft','bitsandbytes']
                       if name in sys.modules]
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(requirements)],check=True)
    # Use a fresh interpreter: imported modules in this kernel may still refer
    # to the packages that pip just replaced or removed.
    smoke = (
        "import torch; "
        "from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, "
        "GenerationConfig; "
        "from peft import PeftModel; "
        "import bitsandbytes; "
        "print('Text evaluation imports: OK; torch=' + torch.__version__)"
    )
    result = subprocess.run([sys.executable,'-c',smoke],stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT,text=True)
    print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Dependency import check failed before evaluation. Full error tail:\n' + result.stdout[-12000:])
    if imported_before:
        raise RuntimeError('Dependencies are repaired, but this session has cached imports. '
                           'Choose Runtime > Restart session (not Disconnect and delete runtime), '
                           'then run this notebook from the top. Drive files and the source checkout remain.')

prepare_text_environment(INFERENCE/'requirements.txt')

## 2. Check the GPU

This evaluation loads Llama in 4-bit NF4, with FP16 computation on T4-class GPUs and native BF16 where supported. This reduces memory requirements but is not a guarantee that every Colab session will fit every prompt.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU assigned. Choose a GPU runtime when one is available.')
props = torch.cuda.get_device_properties(0)
dtype = 'bfloat16' if torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported() else 'float16'
print({'GPU': props.name, 'VRAM_GiB': round(props.total_memory / 2**30, 1), 'dtype': dtype})

## 3. Mount Drive and select the saved run

The default path matches your completed training run. **Do not upload anything if the adapter is already there.** If the run is missing, the next cell offers an optional upload of the exported ZIP and extracts it into a new Drive folder. Extraction verifies the exact adapter checksum and rejects unsafe archive paths.

This notebook never overwrites an existing training run. If you move or rename the run, adjust `ARTIFACT_DIR` to its actual location.

In [ ]:
from google.colab import userdata, drive, files
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Add HF_TOKEN in Colab Secrets and enable notebook access. Do not paste it into source.') from exc
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is empty. Set it in Colab Secrets.')
drive.mount('/content/drive')
sys.path.insert(0, str(INFERENCE))
from adapter_artifact import extract_export, EXPECTED_ADAPTER_SHA256
ARTIFACT_DIR = Path('/content/drive/MyDrive/HealthBrowse/llama3-general-research-002')
if not (ARTIFACT_DIR/'adapter/adapter_model.safetensors').is_file():
    print('Saved adapter not found. Select the exported Llama-3-HealthBrowse-General-QLoRA.zip.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Select exactly one exported adapter ZIP.')
    name, data = next(iter(uploaded.items()))
    if Path(name).name != name or not name.lower().endswith('.zip'):
        raise ValueError('Expected one ZIP filename without path components.')
    imported = Path('/content/drive/MyDrive/HealthBrowse/llama3-general-research-002-imported')
    with tempfile.TemporaryDirectory(prefix='hb-export-') as folder:
        archive = Path(folder)/'adapter-export.zip'
        archive.write_bytes(data)
        extract_export(archive, imported)
    ARTIFACT_DIR = imported
    del uploaded, data
print('Using saved artifact:', ARTIFACT_DIR)
print('Expected adapter SHA256:', EXPECTED_ADAPTER_SHA256)

## 4. Validate inputs before loading the model

Keep `EVALUATION_NAME` unchanged to resume this exact evaluation. If you intentionally change the adapter, prompts, code or settings, use a new evaluation name. A mismatch stops rather than mixing results. Validation records configuration and checksums; it does not run the model.

The original 25-example test split is reported separately from the development questions. If evaluation findings guide later training, keep a new untouched final test set for that later study.

In [ ]:
EVALUATION_NAME = 'evaluation-production-v1'
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,80}', EVALUATION_NAME):
    raise ValueError('Use a simple evaluation folder name.')
EVAL_DIR = ARTIFACT_DIR/EVALUATION_NAME
cmd = [sys.executable, str(INFERENCE/'evaluate_adapter.py'),
       '--artifact', str(ARTIFACT_DIR), '--output', str(EVAL_DIR), '--include-holdout']
subprocess.run(cmd + ['--validate-only'], check=True)
print('Evaluation output folder:', EVAL_DIR)

## 5. Run the evaluation

The model and tokenizer download from Hugging Face using your existing access; no paid inference API is called. Finished results are saved after each case. You can rerun this cell after an interruption with the same settings; completed cases are skipped. If Colab ends the runtime, reopen this notebook and run from the top using the same Drive folder.

Output streams below and is saved in `logs/`. An error includes the underlying output tail, rather than only `CalledProcessError`. This notebook does not keep Colab alive, create a tunnel, or expose a public endpoint.

In [ ]:
from collections import deque
from datetime import datetime, timezone
import importlib.metadata
import uuid

def run_evaluation(command, log_dir):
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True,exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    log_path = log_dir / ('evaluation-' + stamp + '-' + uuid.uuid4().hex[:8] + '.log')
    packages = {}
    for name in ['torch','transformers','accelerate','peft','bitsandbytes','torchvision']:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = 'not installed'
    tail = deque(maxlen=80)
    secrets = [os.environ[name] for name in ['HF_TOKEN','HUGGING_FACE_HUB_TOKEN'] if os.environ.get(name)]
    def redact(text):
        for value in secrets:
            text = text.replace(value,'[REDACTED]')
        return text
    print('Evaluation log:',log_path)
    with log_path.open('w',encoding='utf-8') as log:
        log.write(json.dumps({'python':sys.version,'packages':packages})+'\n')
        log.flush()
        process = subprocess.Popen([command[0],'-u',*command[1:]],stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        try:
            for line in process.stdout:
                line = redact(line)
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
                tail.append(line)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
    if returncode:
        details = ''.join(tail)[-12000:] or '(The process produced no output.)'
        raise RuntimeError(f'Evaluation exited with code {returncode}. Log: {log_path}\n'
                           'The actual evaluation output is below. Share the traceback, not just the exit code.\n'
                           + details)
    return log_path


EVALUATION_LOG = run_evaluation(cmd, EVAL_DIR/'logs')
if not (EVAL_DIR/'summary.json').is_file():
    raise RuntimeError('No summary was produced. Inspect the evaluation log.')
print('Evaluation complete. Results:', EVAL_DIR)

## 6. Read the results

Compare `generation` results for real browser/buddy prompts separately from `holdout-extraction`. `review` controls check a different JSON contract. These scores check structure, source IDs, quote matching and coverage; they do **not** prove that claims fully follow from their sources or that an answer is medically safe.

A low loss or valid JSON cannot repair absent evidence. Review raw answers for direct question answering, relevance, omissions, readability and unsupported statements. Clinical release approval remains false. The source-review packet is included in the exported results, with review fields intentionally blank.

In [ ]:
summary = json.loads((EVAL_DIR/'summary.json').read_text())
print(json.dumps(summary, indent=2))
results = [json.loads(line) for line in (EVAL_DIR/'results.jsonl').read_text().splitlines() if line.strip()]
manifest = json.loads((EVAL_DIR/'evaluation-manifest.json').read_text())
print({'saved_model_results': len(results), 'expected_model_results': 78,
       'clinical_release_approved': manifest['clinical_release_approved']})
for row in results:
    if row['suite'] == 'generation':
        print(row['case_id'], row['model_route'], row['checks'])

## 7. Download and share the evaluation report

This ZIP contains evaluation results, configuration, fixtures and the source-review packet. It does **not** contain model weights, tokens, training checkpoints or your Drive's other files. Logs are kept in Drive for troubleshooting and are not included in the shareable ZIP.

Upload this evaluation ZIP in the HealthBrowse conversation so we can inspect the actual results before selecting the model route. Connecting the public website additionally requires an authorized GPU host with an authenticated HTTPS endpoint. This notebook does not deploy that host.

In [ ]:
import zipfile
archive = EVAL_DIR/'HealthBrowse-Adapter-Evaluation.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for filename in ['evaluation-manifest.json', 'results.jsonl', 'summary.json']:
        path = EVAL_DIR/filename
        if not path.is_file():
            raise RuntimeError('Missing result file: ' + filename)
        zipped.write(path, filename)
    for filename in ['manifest.json', 'production-cases.jsonl', 'review-packet.jsonl']:
        zipped.write(INFERENCE/'evaluation'/filename, 'evaluation-fixtures/'+filename)
files.download(str(archive))

## After evaluation

Follow [the private endpoint guide](https://github.com/McChandra/medical_chatbot/blob/feature/responsive-healthbrowse-auth/healthbrowse/inference/private-endpoint.md) for the optional authenticated, 4-bit Docker service. It supports separate adapter and base-review aliases. End-to-end gateway tests and appropriate clinical review are still required before clinical use.

References: [Meta Llama 3 model and access terms](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct), [PEFT adapters](https://huggingface.co/docs/peft/v0.17.0/en/package_reference/peft_model), [4-bit quantization](https://huggingface.co/docs/transformers/v4.57.1/quantization/bitsandbytes), [Colab FAQ](https://research.google.com/colaboratory/faq.html).

Built with Meta Llama 3. Research only; no clinical certification or guaranteed factuality.